In [1]:
import sys
import plotly.graph_objects as go
from pathlib import Path
import pandas as pd
import numpy as np
import re
# --- Data Loading ---

files = [
    'prices_round_4_day_1.csv',
    'prices_round_4_day_2.csv',
    'prices_round_4_day_3.csv'
]

prices = []
for file in files:
    try:
        df = pd.read_csv(file, sep=';')
        day_match = re.search(r'day_(-?\d+)', file)
        if day_match:
            df['day'] = int(day_match.group(1))
        prices.append(df)
    except FileNotFoundError:
        print(f"Warning: {file} not found.")

df_total = pd.concat(prices, ignore_index=True)
df_total = df_total.sort_values(by=['day', 'timestamp']).reset_index(drop=True)

In [2]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Configuration ──────────────────────────────────────────────────────────────
POSITION_LIMIT = 200        # max abs position
HALF_SPREAD    = 10      # cost per side per unit (adjust to your product)
products       = ["HYDROGEL_PACK"]
days           = [1, 2, 3]
# ───────────────────────────────────────────────────────────────────────────────


def oracle_dp(prices: np.ndarray, half_spread: float, limit: int):
    """
    Backward-induction DP to find the globally optimal trade sequence.

    State : (timestep t, position j) where j ∈ [-L, L] stored offset by L
    Value : maximum *cash* achievable from timestep t onward,
            NOT including mark-to-market on the open position
            (that is added at backtrack / terminal liquidation).

    At each step we may trade any integer Δ ∈ [-L-pos, L-pos],
    crossing the spread on every share transacted.

    Returns
    -------
    trades   : list[int]  — signed trade size at each timestep (0 = hold)
    max_pnl  : float      — realised PnL (with terminal flat liquidation)
    pnl_path : np.ndarray — cumulative PnL at each step after backtracking
    """
    n     = len(prices)
    L     = limit
    n_pos = 2 * L + 1          # index 0 … 2L  ↔  position -L … +L

    NEG_INF = -np.inf
    dp      = np.full((n, n_pos), NEG_INF)
    par     = np.zeros((n, n_pos), dtype=np.int16)   # parent delta

    dp[0][L] = 0.0             # start flat, zero cash

    for t in range(1, n):
        p = prices[t]
        hs = half_spread
        for j in range(n_pos):
            pos = j - L
            best_val  = NEG_INF
            best_delta = 0
            # Try every valid trade size from the PREVIOUS state
            for delta in range(-L - pos, L - pos + 1):
                prev_j = j - delta          # where we were before trading
                if prev_j < 0 or prev_j >= n_pos:
                    continue
                prev_val = dp[t-1][prev_j]
                if prev_val == NEG_INF:
                    continue
                # Cash change: pay ask (p+hs) per unit bought, receive bid (p-hs) per unit sold
                if delta > 0:
                    cash = -delta * (p + hs)
                elif delta < 0:
                    cash = -delta * (p - hs)   # delta<0 → -delta>0, net positive cash
                else:
                    cash = 0.0
                val = prev_val + cash
                if val > best_val:
                    best_val   = val
                    best_delta = delta
            dp[t][j]  = best_val
            par[t][j] = best_delta

    # Terminal: liquidate remaining position at mid (no extra spread for simplicity,
    # or use (p-hs)*pos if you want to penalise the final unwind)
    terminal = np.full(n_pos, NEG_INF)
    p_last   = prices[-1]
    for j in range(n_pos):
        if dp[n-1][j] == NEG_INF:
            continue
        pos = j - L
        # liquidate: sell pos shares at (p_last - hs) if long, buy -pos at (p_last + hs) if short
        liq = pos * (p_last - hs) if pos > 0 else -pos * -(p_last + hs) if pos < 0 else 0
        terminal[j] = dp[n-1][j] + liq

    # Backtrack
    best_end = int(np.argmax(terminal))
    max_pnl  = terminal[best_end]

    trades_rev = []
    j = best_end
    for t in range(n - 1, 0, -1):
        d = int(par[t][j])
        trades_rev.append(d)
        j -= d
    trades_rev.append(0)          # t=0: no incoming trade
    trades = trades_rev[::-1]

    # Reconstruct cumulative PnL step-by-step for the chart
    position   = 0
    cash       = 0.0
    pnl_path   = np.zeros(n)
    for t in range(n):
        d = trades[t]
        if d > 0:
            cash -= d * (prices[t] + half_spread)
        elif d < 0:
            cash += (-d) * (prices[t] - half_spread)
        position += d
        pnl_path[t] = cash + position * prices[t]   # mark-to-market PnL

    return trades, max_pnl, pnl_path


# ── Per-product, per-day analysis & visualisation ─────────────────────────────
for day in days:
    for product in products:
        subset = df_total[
            (df_total['product'] == product) & (df_total['day'] == day)
        ].copy().sort_values('timestamp').reset_index(drop=True)

        if subset.empty:
            print(f"No data for {product} day {day}")
            continue

        subset['mid_price'] = subset['mid_price'].replace(0, np.nan).ffill()
        prices_arr = subset['mid_price'].to_numpy(dtype=float)
        timestamps = subset['timestamp'].to_numpy()

        trades, max_pnl, pnl_path = oracle_dp(prices_arr, HALF_SPREAD, POSITION_LIMIT)
        trades_arr = np.array(trades)

        # ── Classify each timestep ────────────────────────────────────────────
        buy_mask  = trades_arr > 0
        sell_mask = trades_arr < 0

        # For annotation: scale marker size to trade size
        buy_sizes  = np.clip(np.abs(trades_arr[buy_mask])  * 4, 8, 24)
        sell_sizes = np.clip(np.abs(trades_arr[sell_mask]) * 4, 8, 24)

        # ── Figure ────────────────────────────────────────────────────────────
        fig = make_subplots(
            rows=2, cols=1,
            shared_xaxes=True,
            vertical_spacing=0.06,
            row_heights=[0.65, 0.35],
            subplot_titles=(
                f"Oracle-Optimal Trades — {product} (Day {day})",
                f"Cumulative PnL  [oracle max = {max_pnl:,.1f}]"
            )
        )

        # Price path (background)
        fig.add_trace(go.Scatter(
            x=timestamps, y=prices_arr,
            mode='lines', name='Mid Price',
            line=dict(color='rgba(120,120,120,0.45)', width=1.2)
        ), row=1, col=1)

        # Buy markers (triangles up, green)
        if buy_mask.any():
            fig.add_trace(go.Scatter(
                x=timestamps[buy_mask],
                y=prices_arr[buy_mask],
                mode='markers',
                name='Buy',
                marker=dict(
                    symbol='triangle-up',
                    size=buy_sizes,
                    color='#00c97a',
                    line=dict(color='#007a49', width=1)
                ),
                text=[f"+{d}" for d in trades_arr[buy_mask]],
                hovertemplate="BUY %{text}<br>Price: %{y:.2f}<br>t=%{x}<extra></extra>"
            ), row=1, col=1)

        # Sell markers (triangles down, red)
        if sell_mask.any():
            fig.add_trace(go.Scatter(
                x=timestamps[sell_mask],
                y=prices_arr[sell_mask],
                mode='markers',
                name='Sell',
                marker=dict(
                    symbol='triangle-down',
                    size=sell_sizes,
                    color='#ff4455',
                    line=dict(color='#aa1122', width=1)
                ),
                text=[f"{d}" for d in trades_arr[sell_mask]],
                hovertemplate="SELL %{text}<br>Price: %{y:.2f}<br>t=%{x}<extra></extra>"
            ), row=1, col=1)

        # PnL curve (bottom panel)
        pnl_color = [
            '#00c97a' if v >= 0 else '#ff4455'
            for v in pnl_path
        ]
        fig.add_trace(go.Scatter(
            x=timestamps, y=pnl_path,
            mode='lines', name='Cumulative PnL',
            line=dict(color='#4488ff', width=1.5),
            fill='tozeroy',
            fillcolor='rgba(68,136,255,0.12)'
        ), row=2, col=1)

        fig.add_hline(y=0, line_dash='dot', line_color='gray', row=2, col=1)

        fig.update_layout(
            height=750,
            template='plotly_white',
            showlegend=True,
            legend=dict(orientation='h', yanchor='bottom', y=1.01, xanchor='right', x=1),
            title_text=f"DP Oracle — {product} | Day {day} | Limit={POSITION_LIMIT} | HalfSpread={HALF_SPREAD}"
        )
        fig.update_yaxes(title_text='Price',     row=1, col=1)
        fig.update_yaxes(title_text='PnL (shells)', row=2, col=1)
        fig.update_xaxes(title_text='Timestamp', row=2, col=1)

        # Trade summary
        n_buys  = int(buy_mask.sum())
        n_sells = int(sell_mask.sum())
        print(f"{product} Day {day}: {n_buys} buys, {n_sells} sells | Oracle PnL = {max_pnl:,.1f}")

        fig.show()


HYDROGEL_PACK Day 1: 317 buys, 283 sells | Oracle PnL = 293,097.0


HYDROGEL_PACK Day 2: 318 buys, 313 sells | Oracle PnL = 269,776.5


HYDROGEL_PACK Day 3: 271 buys, 291 sells | Oracle PnL = 278,347.5
